# BCC: modern dense-model edit certificate

**Optional paper replication.** This tests a BCC output-head edit family on **SmolLM2-1.7B** (default) or **Qwen2.5-1.5B**, at **1,024-token context**. It measures whether a fixed family of simultaneous sign flips, pruning, and magnitude reductions preserves predictions, and compares it with equal-size magnitude and activation-cost controls.

**Run:** upload this notebook to Google Colab, choose **Runtime → Change runtime type → A100 GPU (40 GB or larger)**, then **Runtime → Run all**. Mount Drive when prompted. Keep the defaults for the SmolLM2 run. Select `qwen25_15b` in the first cell to run Qwen in a separate fresh session. One model runs per session.

Backbone weights and activations stay BF16; the certified output head is a separate calibrated W8/H16 integer head. Primary cache formats are K8/V8 and K8/V4. The source is embedded, so no separate code upload is required.

Each session allows up to 10.5 hours, including an export reserve. A separate timing pilot decides whether the fixed main experiment fits. Passing the scientific criteria is determined by the new data.

**Return:** the final cell downloads the small `BCC_review_*.zip`. It is verified and saved to Drive before the full replay archive, and the retrieval cell works after a runtime reset.

In [ ]:
#@title 1. Select the modern model (SmolLM2 recommended)
MODEL = "smollm2_17b" #@param ["smollm2_17b", "qwen25_15b"]
SAVE_TO_DRIVE = True #@param {type:"boolean"}
JOB_TAG = "" #@param {type:"string"}

import base64, hashlib, io, json, os, re, sys, time, zipfile
from pathlib import Path
from datetime import datetime, timezone

BCC_RELEASE = "1.4.0"
COLAB_PROFILE = "modern-dense-head-v1"
print("BCC COLAB RELEASE", BCC_RELEASE, "|", COLAB_PROFILE)
try:
    import google.colab
except ImportError as exc:
    raise RuntimeError("Open this notebook in Google Colab and select an A100 GPU.") from exc

if not JOB_TAG:
    JOB_TAG = "modern_head_v1_" + MODEL + "_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
if not re.fullmatch(r"modern_head_v1_[A-Za-z0-9_-]{1,64}", JOB_TAG):
    raise ValueError("Leave JOB_TAG blank, or use modern_head_v1_ followed by letters, digits, _ or -.")
BASE = Path("/content/BCC_colab_modern_head_v1")
JOB = BASE / "jobs" / JOB_TAG
DRIVE_DIR = ""
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = str(Path("/content/drive/MyDrive/BCC_results") / JOB_TAG)
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

SOURCE_SHA = "7b3661d4220b8617cddce51c2cc7a678a18bb8d9b44deceb2e9846d55ec9679a"
EMBEDDED_SOURCE = "..."
payload = base64.b64decode(EMBEDDED_SOURCE)
if hashlib.sha256(payload).hexdigest() != SOURCE_SHA:
    raise RuntimeError("Embedded source checksum mismatch.")
CODE = BASE / ("code_" + SOURCE_SHA[:12])
CODE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for item in archive.infolist():
        target = (CODE / item.filename).resolve()
        if not target.is_relative_to(CODE.resolve()):
            raise RuntimeError("Invalid embedded archive path.")
        content = archive.read(item)
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists() and target.read_bytes() != content:
            raise RuntimeError("Extracted source was changed. Start a fresh runtime.")
        if not target.exists():
            target.write_bytes(content)
JOB.mkdir(parents=True, exist_ok=True)
main_config = json.loads((CODE / f"configs/{MODEL}_head_colab.json").read_text())
assert main_config["architecture"] in ("qwen2", "llama")
assert main_config["probe_prefixes"] == 8192
assert main_config["context_lengths"] == [1024]
print("Model:", main_config["model_id"])
print("Pinned revision:", main_config["model_revision"])
print("Main seed:", main_config["seed"], "| control seed:", main_config["selection"]["control_seed"])
print("Probes:", main_config["probe_prefixes"], "| context:", main_config["context_lengths"])
print("Job:", JOB)
print("Drive folder:", DRIVE_DIR or "disabled; download before the runtime is deleted")
print("Verified source:", SOURCE_SHA)


## Frozen experiment

- Same pinned base checkpoints as the previous modern-cache audit; new fixed random streams.
- WikiText-103 finite test-token population; independent uniform endpoint draws with replacement.
- **512 construction, 1,024 calibration, 8,192 probe prefixes**; **1,024 context tokens**, four teacher-forced and four greedy continuation steps.
- BF16 upstream computation and a separately represented W8/H16 output head. Head fitting and candidate selection use separate construction subsets; mask calibration finishes before probing.
- BASE K16/V16 plus primary K8/V8 and K8/V4 deployments. Each edit family is compared with that deployment's own unedited predictions.
- **Matched controls:** magnitude, activation cost (`abs(aligned code) × max(abs(feature))`), and random banks, each with exactly the selected number of editable nonzero entries. All outcomes are exported.
- **Seven criteria:** D ≥ 1,024; risk upper ≤ 1%; added clipped-NLL upper ≤ 0.1 nat/token; population clipped-NLL upper ≤ 95% of uniform-predictor loss; signed head-quality upper and raw mean ≤ 0.1 nat/token; signed deployment-quality upper and raw mean ≤ 0.25 nat/token; packed cache saving ≥ 40%. The loss cap is 20 nats.
- **Confidence:** 168 entries per model, including the union across cache settings. Delta = 1/120 per model and 1/20160 per entry. Joint confidence is 99.166667% for one model and at least 98.333333% across these two planned runs. Earlier experiments have their own budgets.

`PASS_CERTIFIED_EDIT_FAMILY` means both primary rows meet all seven criteria. `PASS_CERTIFIED_FAMILY_WITH_BASELINE_ADVANTAGE` additionally requires positive simultaneous lower risk-reduction bounds against both magnitude and activation-cost controls. The family contains 2^D sign heads; its construction and verification do not enumerate those combinations.

This study provides an output-head edit-family certificate. The saved code and features define its numerical implementation, finite text population and continuation horizon. A whole-checkpoint population-risk bound is a separate claim.

In [ ]:
#@title 2. Run the full experiment (up to 10.5 hours including export)
import subprocess
if globals().get("COLAB_PROFILE") != "modern-dense-head-v1" or globals().get("SOURCE_SHA") != "7b3661d4220b8617cddce51c2cc7a678a18bb8d9b44deceb2e9846d55ec9679a":
    raise RuntimeError("Run the setup cell from this notebook first.")
cmd = [sys.executable, str(CODE / "colab/run_colab.py"),
       "--job", str(JOB), "--model", MODEL, "--budget-hours", "10.5"]
if DRIVE_DIR:
    cmd += ["--drive-dir", DRIVE_DIR]
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    returncode = process.wait()
except KeyboardInterrupt:
    import signal
    process.send_signal(signal.SIGINT)
    print("Stopping computation and exporting available evidence. Please wait.")
    for line in process.stdout:
        print(line, end="", flush=True)
    returncode = process.wait()
print("Controller exit code:", returncode)
print("Run the next cell to view the report and download the small review ZIP.")


## Download and preserve the results

Send back the **small `BCC_review_*.zip`**; retain the full `BCC_results_*.zip` in Drive. The review includes all available final tables, controls, head quality, frozen confidence ledger and status. `complete_main_report` distinguishes completed reports from partial-run diagnostics.

If automatic download is blocked, open **Drive → MyDrive → BCC_results → your modern_head_v1 folder** and download the review ZIP.

**After a runtime reset:** run the final code cell by itself. Enter a specific Drive run folder in `REVIEW_FOLDER`, or leave it blank to retrieve the latest saved modern dense-model review. It does not rerun inference.

In [ ]:
#@title 3. Download review ZIP (also works after a runtime reset)
REVIEW_FOLDER = "" #@param {type:"string"}
DOWNLOAD_FULL_ARCHIVE = False #@param {type:"boolean"}

from pathlib import Path
import hashlib, json, zipfile
from IPython.display import display, Markdown
from google.colab import files, drive

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

folders = []
if REVIEW_FOLDER.strip():
    folders = [Path(REVIEW_FOLDER.strip())]
else:
    current_job = globals().get("JOB")
    if current_job is not None:
        folders.append(Path(current_job))
    if globals().get("DRIVE_DIR"):
        folders.append(Path(DRIVE_DIR))

# Mount only if a Drive search is needed and it is not already mounted.
if not folders or any(str(p).startswith("/content/drive/") for p in folders):
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
if not folders:
    root = Path("/content/drive/MyDrive/BCC_results")
    folders = [p for p in sorted(root.glob("modern_head_v1_*"), key=lambda p: p.name[-15:], reverse=True)
               if p.is_dir() and list(p.glob("BCC_review_*.zip"))][:1]
    if folders:
        print("Retrieving newest saved modern dense-model review:", folders[0])

ZIP_PATH = None
location = {}
chosen_folder = None
for folder in folders:
    record = folder / "REVIEW_LOCATION.json"
    loc = json.loads(record.read_text()) if record.is_file() else {}
    candidates = [Path(loc[k]) for k in ("local_zip", "drive_zip") if loc.get(k)]
    candidates += sorted((folder / "exports").glob("BCC_review_*.zip"), reverse=True)
    candidates += sorted(folder.glob("BCC_review_*.zip"), reverse=True)
    for candidate in candidates:
        if candidate.is_file():
            ZIP_PATH, location, chosen_folder = candidate, loc, folder
            break
    if ZIP_PATH is not None:
        break
if ZIP_PATH is None:
    raise FileNotFoundError("No small review ZIP was found for this run. Check the run cell output and its Drive folder. If only BCC_checkpoint.zip exists, keep that checkpoint and share the log; the main report may not have finished.")

actual_sha = file_sha256(ZIP_PATH)
if location.get("sha256") and actual_sha != location["sha256"]:
    raise RuntimeError("Review ZIP checksum mismatch. Keep the file and controller log for inspection.")
with zipfile.ZipFile(ZIP_PATH) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("Review ZIP failed its integrity check.")
    for entry in json.loads(archive.read("MANIFEST.json")):
        content = archive.read(entry["path"])
        if len(content) != entry["bytes"] or hashlib.sha256(content).hexdigest() != entry["sha256"]:
            raise RuntimeError("Review manifest mismatch: " + entry["path"])
    review_status = json.loads(archive.read("REVIEW_STATUS.json"))
    print(json.dumps(review_status, indent=2))
    if "runs/main/REPORT.md" in archive.namelist():
        display(Markdown(archive.read("runs/main/REPORT.md").decode()))
    elif "TIME_FORECAST.json" in archive.namelist():
        print(archive.read("TIME_FORECAST.json").decode())
print("Verified small ZIP:", ZIP_PATH)
print("Size (MiB):", round(ZIP_PATH.stat().st_size / 2**20, 2))
print("SHA256:", actual_sha)
print("Send back:", ZIP_PATH.name)
files.download(str(ZIP_PATH))

full_location = {}
for folder in [chosen_folder] + folders:
    record = folder / "RESULT_LOCATION.json"
    if record.is_file():
        full_location = json.loads(record.read_text())
        break
if full_location:
    print("Full archive export status:", full_location.get("export_status", "unknown"))
    print("Full archive in Drive:", full_location.get("drive_zip") or "No verified Drive copy recorded.")
    if full_location.get("export_error"):
        print("Full archive export message:", full_location["export_error"])
if DOWNLOAD_FULL_ARCHIVE:
    full_path = next((Path(full_location[k]) for k in ("local_zip", "drive_zip")
                      if full_location.get(k) and Path(full_location[k]).is_file()), None)
    if full_path is None:
        raise FileNotFoundError("A completed full archive is not available. Keep the review ZIP and the latest checkpoint.")
    if full_location.get("sha256") and file_sha256(full_path) != full_location["sha256"]:
        raise RuntimeError("Full archive checksum mismatch.")
    print("Downloading full evidence archive:", full_path)
    files.download(str(full_path))


## Validation and interpretation

The certificate and confidence-bound implementation are inherited from the OLMoE run. New adapter tests cover native BF16 parity, exact integer scores, cache packing and isolation of the editable head from tied input embeddings. Both architectures are exercised through complete CPU software fixtures and result export. **Public-checkpoint GPU runs have not been performed while preparing this notebook.**

The strongest new evidence would be a certified family with a positive simultaneous advantage over the activation-cost control at this longer context. The complete report retains all other outcomes. A successful result can be added as an appendix replication row.

The full archive contains frozen settings, head-selection records, masks, features, witnesses, code, timings and environment records. Public weights are referenced by immutable revisions and hashes. See `code/docs/MODERN_DENSE_HEAD.md` for the protocol. Replay a full archive with:

```bash
python -m pip install -r code/requirements.txt
python code/bcc.py verify --run runs/main --device cpu
```

Official model pages: [SmolLM2-1.7B](https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B), [Qwen2.5-1.5B](https://huggingface.co/Qwen/Qwen2.5-1.5B).